# Flood Risk Prediction

Welcome to this end-to-end notebook demonstrating the full Machine Learning lifecycle for predicting flood probaility.

**What we shall do:**
- Problem Definition & Motivation
- Data Cleaning & Preprocessing
- Feature Engineering & Selection
- Exploratory Data Analysis (EDA) with visualizations
- Model Selection & Baseline Comparisons
- Model Training, Evaluation, and Interpretation

In [ ]:
# ## 1. Imports & Environment Setup
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from pycaret.regression import *
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Problem Definition

**Objective:** Build a regression model to predict flood probability in a region.  

**Why Regression?**  
- Directly quantifies flood-driving volume.  
- Enables threshold-based classification for flood alerts.  
- Simpler and more interpretable than direct classification on limited data.  

**Target:** `FloodPrediction` that is in continous form.  

**Features** 
- MonsoonIntensity
- TopographyDrainage
- RiverManagement
- Deforestation
- Urbanization
- ClimateChange
- DamsQuality
- Siltation
- AgriculturalPractices
- Encroachments
- IneffectiveDisasterPreparedness
- DrainageSystems
- CoastalVulnerability
- Landslides
- Watersheds
- DeterioratingInfrastructure
- PopulationScore
- WetlandLoss
- InadequatePlanning
- PoliticalFactors

## 3. Data Acquisition, Cleaning and Preprocessing

We managed to acquire a dataset from kaggle.com with sufficient information to train our model. It seemed well processed already but we decided to have our own take on it, for the sake of research and understanding the underlying principles of handling datasets for ML.

In [ ]:
df = pd.read_csv('data/flood.csv')

# Drop rows with NaN or infinite values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

# Check for class imbalance in target (though continuous)
sns.kdeplot(df['FloodProbability'])
plt.title('Target Variable Distribution')
plt.legend(['Flood Probability'])
plt.show()

In [ ]:
import smogn

# Ensure all columns are float32 for compatibility and efficiency with smogn
df = df.astype('float32')

relevance_config = np.array([
    [0.4, 1.0, 0.5],
    [0.5, 0.0, 0.5],
    [0.6, 1.0, 0.5]
])

# 2. Apply SMOGN with custom relevance
df_smogn = smogn.smoter(
    data=df,
    y='FloodProbability',
    rel_thres=0.8,
    rel_method='manual',
    rel_ctrl_pts_rg=relevance_config,
    samp_method='extreme',
    k=3
)

print("Original shape:", df.shape)
print("SMOGN-balanced shape:", df_smogn.shape)

# Use the new dataframe for training
df = df_smogn.copy()

sns.kdeplot(df['FloodProbability'])
plt.title('Target Variable Distribution')
plt.show()

### 4. Feature Engineering and Selection

In [ ]:
# 1. Interaction terms (domain knowledge-based)
df['Monsoon_Drainage_Interaction'] = df['MonsoonIntensity'] * df['DrainageSystems']
df['Urbanization_Encroachment'] = df['Urbanization'] * df['Encroachments']

# 2. Climate vulnerability index
climate_features = ['ClimateChange', 'CoastalVulnerability', 'WetlandLoss']
df['ClimateVulnerability'] = df[climate_features].mean(axis=1)

# 3. Infrastructure quality composite
infra_features = ['RiverManagement', 'DamsQuality', 'DrainageSystems']
df['InfrastructureQuality'] = df[infra_features].mean(axis=1)

X = df.drop('FloodProbability', axis=1)
y = df['FloodProbability']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 5. Exploratory Data Analysis (EDA)

Let's visualize:
1. Basic Summary Statistics
2. Missing Values Check
3. Distribution Plots
4. Correlation Matrix
5. Pairplots / Scatterplots
Focused on top-correlated features with FloodProbability
Helps visually identify linear/nonlinear trends
6. Boxplots
Show how each feature influences FloodProbability
Detect outliers (e.g., via IQR)

In [ ]:
print("Dataset Description:\n")
#print(df.info())
print(df.describe())

# Check missing values
print("MISSING VALUES:\n", df.isnull().sum())

# Verify feature distributions
df.hist(figsize=(20, 15), bins=10)
plt.tight_layout()
plt.show()

# Visualize sampled target variable distribution
sns.kdeplot(df['FloodProbability'])
plt.figure(figsize=(10, 6))
plt.title('Target Variable Distribution')
plt.legend(['Flood Probability'])
plt.show()

# ====== CORRELATION ANALYSIS ======
plt.figure(figsize=(20, 18))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".1f", cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

target_corr = corr_matrix['FloodProbability'].abs().sort_values(ascending=False)
print("Top features by correlation:\n", target_corr.head(10))

# ======= PAIRPLOT ANALYSIS =======
# Limited to a few relevant features to avoid clutter
selected_cols = ['InfrastructureQuality', 'ClimateVulnerability', 'Urbanization_Encroachment', 'Monsoon_Drainage_Interaction','DeterioratingInfrastructure', 'TopographyDrainage','RiverManagement','Watersheds','DamsQuality']
df['FloodRiskLevel'] = pd.cut(df['FloodProbability'], bins=3, labels=["Low", "Medium", "High"])

sns.pairplot(df[selected_cols + ['FloodRiskLevel']], hue='FloodRiskLevel', palette='coolwarm')
plt.show()

# ======= BOXPLOT ANALYSIS =======

for feature in selected_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=feature, y='FloodProbability', data=df)
    plt.title(f'FloodProbability vs {feature}')
    plt.tight_layout()
    plt.show()

### 6. Model Selection & Baseline Comparisons
